# Stuttering Detection: Tree-Based Ensemble Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Single Decision Trees vs. Random Forests

---

## Step 1: Initialization & Environment
We use the project's modular engine to load the 768-dimensional WavLM embeddings and prepare a fair, balanced experimental setup.

In [5]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import DecisionTreeModel, RandomForestModel

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [9]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 10000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

[System] Skipping extraction. Using existing data on disk.


## Step 3: Data Preparation Engine
We load our distributed features, filter for quality, and use **Oversampling** to handle the class imbalance. Unlike Neural Networks, tree-based models are robust and do not strictly require standardization, but we apply it for consistency.

In [6]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features into matrices
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Handle Class Imbalance
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Setting these to variables for training clarity
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Decision Tree
The Decision Tree is a single 'Logical Flow' that splits the dataset based on feature thresholds. While intuitive, a single tree is prone to **Overfitting** high-dimensional audio features.

In [7]:
dt_model = DecisionTreeModel("Decision_Tree_Baseline", max_depth=10)
dt_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
dt_model.evaluate(X_test_final, y_test)

[Model: Decision_Tree_Baseline] Initialized.
[Decision_Tree_Baseline] Building Logic Tree (Max Depth: 10)...

--- Evaluation on Unseen Test Set ---

--- Evaluation: Decision_Tree_Baseline ---
Accuracy: 0.5857
Precision: 0.4216
Recall: 0.4264
F1: 0.4240

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      321             155            
True: Stutter(1)     152             113            


{'accuracy': 0.5856950067476383,
 'precision': 0.4216417910447761,
 'recall': 0.42641509433962266,
 'f1': 0.42401500938086306,
 'confusion_matrix': array([[321, 155],
        [152, 113]])}

## Step 5: Model 2 - Random Forest (Ensemble)
Random Forest is an ensemble method that trains **100 independent decision trees** on different subsets of the data. By averaging their results, it overcomes the overfitting issues of a single tree, providing a much more robust boundary for stuttering detection.

In [8]:
rf_model = RandomForestModel("Random_Forest_Ensemble", n_estimators=100, max_depth=15)
rf_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
rf_model.evaluate(X_test_final, y_test)

[Model: Random_Forest_Ensemble] Initialized.
[Random_Forest_Ensemble] Planting 100 Trees (Max Depth: 15)...

--- Evaluation on Unseen Test Set ---

--- Evaluation: Random_Forest_Ensemble ---
Accuracy: 0.6829
Precision: 0.6087
Recall: 0.3170
F1: 0.4169

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      422             54             
True: Stutter(1)     181             84             


{'accuracy': 0.6828609986504723,
 'precision': 0.6086956521739131,
 'recall': 0.3169811320754717,
 'f1': 0.41687344913151364,
 'confusion_matrix': array([[422,  54],
        [181,  84]])}

## Step 6: Comparative Results
By observing the results, we can confirm the 'Wisdom of the Crowd.'

**Finding**: The Random Forest provides a smoother generalization. While the Decision Tree might memorize specific audio samples, the Forest learns the underlying acoustic structure of disfluent speech, making it more reliable for real-world audio from new podcasts.